# 🎮 VALORANT Personal Coach

## Notebook 02 — Model Training and Personalized Agent Ranking

This notebook trains and evaluates three machine-learning models for the
personalized VALORANT agent recommendation system:

1. Logistic Regression
2. Random Forest Classifier
3. CatBoost Classifier

The models are trained using historical features generated only from matches
played before each target match.

The supporting prediction task estimates:

**P(Win | Map, Candidate Agent, Historical Player Features)**

The predicted probabilities are then used as recommendation scores to rank the
player's previously used agents for a selected map.

The final system returns the player's **Top 5–6 recommended agents** rather than
only predicting a Win or Loss.

In [1]:
%pip install -q scikit-learn catboost joblib

Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from catboost import CatBoostClassifier

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier

In [3]:
# ============================================================
# PROJECT PATHS
# ============================================================

CURRENT_DIR = Path.cwd()

# Works whether notebook is launched from:
# valorant_personal_coach/
# or
# valorant_personal_coach/notebooks/

if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR


PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
RESULTS_DIR = PROJECT_ROOT / "results"

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)


FEATURE_PATH = (
    PROCESSED_DIR
    / "competitive_matches_features.csv"
)

CLEANED_PATH = (
    PROCESSED_DIR
    / "competitive_matches_cleaned.csv"
)


print("Project root:")
print(PROJECT_ROOT)

print("\nFeature dataset:")
print(FEATURE_PATH)

print("\nCleaned dataset:")
print(CLEANED_PATH)

Project root:
c:\Users\HP\Desktop\valorant_personal_coach

Feature dataset:
c:\Users\HP\Desktop\valorant_personal_coach\data\processed\competitive_matches_features.csv

Cleaned dataset:
c:\Users\HP\Desktop\valorant_personal_coach\data\processed\competitive_matches_cleaned.csv


In [4]:
# ============================================================
# LOAD DATA
# ============================================================

features_df = pd.read_csv(
    FEATURE_PATH,
    parse_dates=["Date"]
)

cleaned_df = pd.read_csv(
    CLEANED_PATH,
    parse_dates=["Date"]
)


features_df = (
    features_df
    .sort_values(["Date", "Match id"])
    .reset_index(drop=True)
)

cleaned_df = (
    cleaned_df
    .sort_values(["Date", "Match id"])
    .reset_index(drop=True)
)


print(
    "Feature dataset shape:",
    features_df.shape
)

print(
    "Cleaned dataset shape:",
    cleaned_df.shape
)

Feature dataset shape: (297, 29)
Cleaned dataset shape: (297, 26)


In [5]:
print("Results:")
print(
    features_df["Result"]
    .value_counts()
)

print(
    "\nWin Flag:"
)

print(
    features_df["Win Flag"]
    .value_counts(
        dropna=False
    )
)

Results:
Result
Loss    149
Win     138
Draw     10
Name: count, dtype: int64

Win Flag:
Win Flag
0.0    149
1.0    138
NaN     10
Name: count, dtype: int64


## Model Input Features

Only information that would have been available before the target match is used
as model input.

Current-match K/D, ACS, kills, deaths, TRS, DDΔ, final scores, and match ending
type are deliberately excluded because using them would introduce data leakage.

The model instead receives historical features such as recent form, previous
agent performance, previous map performance, Agent × Map history, familiarity,
and recency.

In [6]:
# ============================================================
# MODEL FEATURES
# ============================================================

categorical_features = [
    "Map",
    "Agent",
    "agent_familiarity"
]


numeric_features = [
    "previous_matches",
    "previous_win_rate",
    "previous_kd",

    "recent_5_win_rate",
    "recent_10_win_rate",
    "recent_5_kd",
    "recent_10_kd",
    "recent_5_acs",
    "recent_10_acs",

    "agent_previous_matches",
    "agent_previous_win_rate",
    "agent_previous_kd",
    "agent_previous_acs",

    "map_previous_matches",
    "map_previous_win_rate",
    "map_previous_kd",

    "agent_map_previous_matches",
    "agent_map_previous_win_rate",
    "agent_map_previous_kd",

    "agent_usage_frequency",
    "agent_recency"
]


model_features = (
    categorical_features
    + numeric_features
)


print(
    "Number of model features:",
    len(model_features)
)

print("\nCategorical:")
print(categorical_features)

print("\nNumerical:")
print(numeric_features)

Number of model features: 24

Categorical:
['Map', 'Agent', 'agent_familiarity']

Numerical:
['previous_matches', 'previous_win_rate', 'previous_kd', 'recent_5_win_rate', 'recent_10_win_rate', 'recent_5_kd', 'recent_10_kd', 'recent_5_acs', 'recent_10_acs', 'agent_previous_matches', 'agent_previous_win_rate', 'agent_previous_kd', 'agent_previous_acs', 'map_previous_matches', 'map_previous_win_rate', 'map_previous_kd', 'agent_map_previous_matches', 'agent_map_previous_win_rate', 'agent_map_previous_kd', 'agent_usage_frequency', 'agent_recency']


In [7]:
# ============================================================
# BINARY MODELLING DATA
# ============================================================

model_df = (
    features_df[
        features_df["Win Flag"].notna()
    ]
    .copy()
    .reset_index(drop=True)
)


X = model_df[
    model_features
].copy()

y = (
    model_df["Win Flag"]
    .astype(int)
)


print(
    "Binary modelling rows:",
    len(model_df)
)

print("\nTarget distribution:")
print(y.value_counts())

Binary modelling rows: 287

Target distribution:
Win Flag
0    149
1    138
Name: count, dtype: int64


## Chronological Data Split

A random split is not used because the project represents sequential gameplay.

Older matches are used to learn player behaviour, while newer matches are kept
for validation and testing.

The chronological split is:

- 70% Training
- 15% Validation
- 15% Testing

This prevents future matches from influencing models trained on earlier
gameplay history.

In [8]:
# ============================================================
# CHRONOLOGICAL SPLIT
# ============================================================

n = len(model_df)

train_end = int(n * 0.70)
validation_end = int(n * 0.85)


train_df = model_df.iloc[
    :train_end
].copy()

validation_df = model_df.iloc[
    train_end:validation_end
].copy()

test_df = model_df.iloc[
    validation_end:
].copy()


X_train = train_df[model_features]
y_train = train_df["Win Flag"].astype(int)

X_validation = validation_df[model_features]
y_validation = validation_df["Win Flag"].astype(int)

X_test = test_df[model_features]
y_test = test_df["Win Flag"].astype(int)


print(
    "Training matches:",
    len(train_df)
)

print(
    "Validation matches:",
    len(validation_df)
)

print(
    "Testing matches:",
    len(test_df)
)


print(
    "\nTraining period:",
    train_df["Date"].min(),
    "to",
    train_df["Date"].max()
)

print(
    "\nValidation period:",
    validation_df["Date"].min(),
    "to",
    validation_df["Date"].max()
)

print(
    "\nTesting period:",
    test_df["Date"].min(),
    "to",
    test_df["Date"].max()
)

Training matches: 200
Validation matches: 43
Testing matches: 44

Training period: 2024-11-07 00:00:00 to 2025-06-22 00:00:00

Validation period: 2025-06-23 00:00:00 to 2025-08-30 00:00:00

Testing period: 2026-08-08 00:00:00 to 2026-09-20 00:00:00


In [9]:
# ============================================================
# PREPROCESSING
# ============================================================

numeric_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)


categorical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)


preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_transformer,
            numeric_features
        ),
        (
            "categorical",
            categorical_transformer,
            categorical_features
        )
    ]
)

In [10]:
# ============================================================
# MODEL 1 — LOGISTIC REGRESSION
# ============================================================

logistic_model = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "classifier",
            LogisticRegression(
                max_iter=2000,
                class_weight="balanced",
                random_state=42
            )
        )
    ]
)


logistic_model.fit(
    X_train,
    y_train
)


print(
    "Logistic Regression trained successfully."
)

Logistic Regression trained successfully.


In [11]:
# ============================================================
# RANDOM FOREST PREPROCESSING
# ============================================================

rf_numeric_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        )
    ]
)


rf_categorical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)


rf_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            rf_numeric_transformer,
            numeric_features
        ),
        (
            "categorical",
            rf_categorical_transformer,
            categorical_features
        )
    ]
)

In [12]:
# ============================================================
# MODEL 2 — RANDOM FOREST
# ============================================================

random_forest_model = Pipeline(
    steps=[
        (
            "preprocessor",
            rf_preprocessor
        ),
        (
            "classifier",
            RandomForestClassifier(
                n_estimators=300,
                max_depth=6,
                min_samples_leaf=4,
                class_weight="balanced",
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)


random_forest_model.fit(
    X_train,
    y_train
)


print(
    "Random Forest trained successfully."
)

Random Forest trained successfully.


In [13]:
# ============================================================
# CATBOOST PREPARATION
# ============================================================

def prepare_catboost_data(
    X,
    numeric_fill_values=None
):
    """
    Prepare a DataFrame for CatBoost while preserving
    categorical columns.
    """

    data = X.copy()

    # Categorical missing values
    for column in categorical_features:

        data[column] = (
            data[column]
            .astype("string")
            .fillna("Unknown")
            .astype(str)
        )

    # Calculate numeric medians using training data only
    if numeric_fill_values is None:

        numeric_fill_values = (
            data[numeric_features]
            .median()
        )

    # Fill historical cold-start NaNs
    data[numeric_features] = (
        data[numeric_features]
        .fillna(numeric_fill_values)
    )

    return data, numeric_fill_values

In [14]:
X_train_cat, cat_numeric_fill = (
    prepare_catboost_data(
        X_train
    )
)


X_validation_cat, _ = (
    prepare_catboost_data(
        X_validation,
        numeric_fill_values=cat_numeric_fill
    )
)


X_test_cat, _ = (
    prepare_catboost_data(
        X_test,
        numeric_fill_values=cat_numeric_fill
    )
)


cat_feature_indices = [
    X_train_cat.columns.get_loc(column)
    for column in categorical_features
]


print(
    "CatBoost categorical feature indices:",
    cat_feature_indices
)

CatBoost categorical feature indices: [0, 1, 2]


In [15]:
# ============================================================
# MODEL 3 — CATBOOST
# ============================================================

catboost_model = CatBoostClassifier(
    iterations=200,
    depth=4,
    learning_rate=0.05,
    loss_function="Logloss",
    random_seed=42,
    verbose=False,
    l2_leaf_reg=3,
    random_strength=0.5
)

catboost_model.fit(
    X_train_cat,
    y_train,

    cat_features=cat_feature_indices,

    eval_set=(
        X_validation_cat,
        y_validation
    ),

    early_stopping_rounds=25,

    verbose=False
)


print(
    "CatBoost trained successfully."
)

print(
    "Best iteration:",
    catboost_model.get_best_iteration()
)

CatBoost trained successfully.
Best iteration: 4


In [16]:
# # ============================================================
# # MODEL 3 — CATBOOST
# # ============================================================

# catboost_model = CatBoostClassifier(
#     iterations=300,
#     depth=5,
#     learning_rate=0.03,
#     loss_function="Logloss",
#     random_seed=42,
#     verbose=False,
#     l2_leaf_reg=5
# )


# catboost_model.fit(
#     X_train_cat,
#     y_train,

#     cat_features=cat_feature_indices,

#     eval_set=(
#         X_validation_cat,
#         y_validation
#     ),

#     early_stopping_rounds=40,

#     verbose=False
# )


# print(
#     "CatBoost trained successfully."
# )

# print(
#     "Best iteration:",
#     catboost_model.get_best_iteration()
# )

In [17]:
# ============================================================
# MODEL PROBABILITY FUNCTION
# ============================================================

def predict_win_probability(
    model_name,
    model,
    X_data
):
    """
    Return P(Win = 1).
    """

    if model_name == "CatBoost":

        prepared, _ = prepare_catboost_data(
            X_data,
            numeric_fill_values=cat_numeric_fill
        )

        return model.predict_proba(
            prepared
        )[:, 1]

    return model.predict_proba(
        X_data
    )[:, 1]

In [18]:
validation_preview = validation_df[
    [
        "Match id",
        "Date",
        "Map",
        "Agent",
        "Result"
    ]
].copy()


validation_preview[
    "Logistic Probability"
] = predict_win_probability(
    "Logistic Regression",
    logistic_model,
    X_validation
)


validation_preview[
    "Random Forest Probability"
] = predict_win_probability(
    "Random Forest",
    random_forest_model,
    X_validation
)


validation_preview[
    "CatBoost Probability"
] = predict_win_probability(
    "CatBoost",
    catboost_model,
    X_validation
)


validation_preview.head(10)

,Match id,Date,Map,Agent,Result,Logistic Probability,Random Forest Probability,CatBoost Probability
200,206,2025-06-23,Pearl,Sage,Win,0.463228,0.414566,0.496258
201,207,2025-06-23,Icebox,Sage,Win,0.443768,0.396050,0.496258
202,208,2025-06-24,Split,Sage,Win,0.449645,0.365895,0.487569
203,209,2025-06-25,Haven,Sage,Win,0.523382,0.390598,0.487850
204,210,2025-06-25,Lotus,Killjoy,Win,0.775676,0.532511,0.553289
205,211,2025-06-25,Split,Sage,Win,0.309219,0.312464,0.487569
206,212,2025-06-25,Haven,Sage,Loss,0.418447,0.345890,0.487850
207,213,2025-06-26,Lotus,Sage,Loss,0.226961,0.332260,0.495479
208,214,2025-06-26,Bind,Reyna,Loss,0.721044,0.478794,0.507320
209,215,2025-06-26,Haven,Sage,Loss,0.466581,0.414511,0.496258


In [19]:
# ============================================================
# HISTORICAL HELPERS
# ============================================================

def historical_win_rate(history):
    """
    Win rate using Win/Loss matches.
    Draw Win Flags are NaN and therefore ignored.
    """

    if len(history) == 0:
        return np.nan

    values = history["Win Flag"].dropna()

    if len(values) == 0:
        return np.nan

    return values.mean()


def historical_kd(history):
    """
    Aggregate K/D:
    total kills / total deaths
    """

    if len(history) == 0:
        return np.nan

    kills = history["Kills"].sum()
    deaths = history["Deaths"].sum()

    if deaths == 0:
        return np.nan

    return kills / deaths


def familiarity_level(matches):

    if matches == 0:
        return "New"

    if matches < 5:
        return "Very Low"

    if matches < 15:
        return "Low"

    if matches < 40:
        return "Medium"

    return "High"

In [20]:
# ============================================================
# BUILD ONE CANDIDATE AGENT PROFILE
# ============================================================

def build_candidate_profile(
    history,
    current_map,
    candidate_agent
):
    """
    Construct the pre-match historical feature profile for
    one candidate agent.
    """

    history = history.copy()

    previous_matches = len(history)


    # --------------------------------------------------------
    # Overall player history
    # --------------------------------------------------------

    previous_win_rate = (
        historical_win_rate(history)
    )

    previous_kd = (
        historical_kd(history)
    )


    # --------------------------------------------------------
    # Recent history
    # --------------------------------------------------------

    recent_5 = history.tail(5)
    recent_10 = history.tail(10)


    recent_5_win_rate = (
        historical_win_rate(recent_5)
    )

    recent_10_win_rate = (
        historical_win_rate(recent_10)
    )


    recent_5_kd = (
        historical_kd(recent_5)
    )

    recent_10_kd = (
        historical_kd(recent_10)
    )


    recent_5_acs = (
        recent_5["ACS"].mean()
        if len(recent_5) > 0
        else np.nan
    )

    recent_10_acs = (
        recent_10["ACS"].mean()
        if len(recent_10) > 0
        else np.nan
    )


    # --------------------------------------------------------
    # Agent history
    # --------------------------------------------------------

    agent_history = history[
        history["Agent"]
        == candidate_agent
    ]


    agent_previous_matches = (
        len(agent_history)
    )


    agent_previous_win_rate = (
        historical_win_rate(
            agent_history
        )
    )


    agent_previous_kd = (
        historical_kd(
            agent_history
        )
    )


    agent_previous_acs = (
        agent_history["ACS"].mean()
        if len(agent_history) > 0
        else np.nan
    )


    # --------------------------------------------------------
    # Map history
    # --------------------------------------------------------

    map_history = history[
        history["Map"]
        == current_map
    ]


    map_previous_matches = (
        len(map_history)
    )


    map_previous_win_rate = (
        historical_win_rate(
            map_history
        )
    )


    map_previous_kd = (
        historical_kd(
            map_history
        )
    )


    # --------------------------------------------------------
    # Agent × Map history
    # --------------------------------------------------------

    agent_map_history = history[
        (
            history["Agent"]
            == candidate_agent
        )
        &
        (
            history["Map"]
            == current_map
        )
    ]


    agent_map_previous_matches = (
        len(agent_map_history)
    )


    agent_map_previous_win_rate = (
        historical_win_rate(
            agent_map_history
        )
    )


    agent_map_previous_kd = (
        historical_kd(
            agent_map_history
        )
    )


    # --------------------------------------------------------
    # Familiarity
    # --------------------------------------------------------

    if previous_matches > 0:

        agent_usage_frequency = (
            agent_previous_matches
            / previous_matches
        )

    else:

        agent_usage_frequency = np.nan


    # --------------------------------------------------------
    # Agent recency
    # --------------------------------------------------------

    if len(agent_history) > 0:

        last_agent_index = (
            agent_history.index[-1]
        )

        current_index = (
            history.index[-1] + 1
        )

        agent_recency = (
            current_index
            - last_agent_index
        )

    else:

        agent_recency = np.nan


    profile = {
        "Map": current_map,
        "Agent": candidate_agent,

        "previous_matches":
            previous_matches,

        "previous_win_rate":
            previous_win_rate,

        "previous_kd":
            previous_kd,

        "recent_5_win_rate":
            recent_5_win_rate,

        "recent_10_win_rate":
            recent_10_win_rate,

        "recent_5_kd":
            recent_5_kd,

        "recent_10_kd":
            recent_10_kd,

        "recent_5_acs":
            recent_5_acs,

        "recent_10_acs":
            recent_10_acs,

        "agent_previous_matches":
            agent_previous_matches,

        "agent_previous_win_rate":
            agent_previous_win_rate,

        "agent_previous_kd":
            agent_previous_kd,

        "agent_previous_acs":
            agent_previous_acs,

        "map_previous_matches":
            map_previous_matches,

        "map_previous_win_rate":
            map_previous_win_rate,

        "map_previous_kd":
            map_previous_kd,

        "agent_map_previous_matches":
            agent_map_previous_matches,

        "agent_map_previous_win_rate":
            agent_map_previous_win_rate,

        "agent_map_previous_kd":
            agent_map_previous_kd,

        "agent_usage_frequency":
            agent_usage_frequency,

        "agent_recency":
            agent_recency,

        "agent_familiarity":
            familiarity_level(
                agent_previous_matches
            )
    }


    return profile

In [21]:
# ============================================================
# CANDIDATE AGENT GENERATION
# ============================================================

def build_candidate_table(
    history,
    current_map,
    min_previous_matches=1
):
    """
    Build one feature row for every agent the player has
    previously used.
    """

    agent_counts = (
        history["Agent"]
        .value_counts()
    )


    candidate_agents = (
        agent_counts[
            agent_counts
            >= min_previous_matches
        ]
        .index
        .tolist()
    )


    profiles = []


    for agent in candidate_agents:

        profile = build_candidate_profile(
            history=history,
            current_map=current_map,
            candidate_agent=agent
        )

        profiles.append(profile)


    return pd.DataFrame(profiles)

In [22]:
# ============================================================
# RANK AGENTS
# ============================================================

def rank_agents(
    model_name,
    model,
    history,
    current_map,
    top_k=6
):
    """
    Rank candidate agents by predicted win probability.
    """

    candidates = build_candidate_table(
        history=history,
        current_map=current_map,
        min_previous_matches=1
    )


    if candidates.empty:

        return candidates


    probabilities = (
        predict_win_probability(
            model_name,
            model,
            candidates[model_features]
        )
    )


    candidates[
        "Predicted Win Probability"
    ] = probabilities


    ranked = (
        candidates[
            [
                "Agent",
                "Predicted Win Probability",
                "agent_familiarity",
                "agent_previous_matches",
                "agent_previous_win_rate",
                "agent_map_previous_matches",
                "agent_map_previous_win_rate"
            ]
        ]
        .sort_values(
            "Predicted Win Probability",
            ascending=False
        )
        .reset_index(drop=True)
    )


    ranked.index = (
        ranked.index + 1
    )

    ranked.index.name = "Rank"


    return ranked.head(top_k)

In [23]:
# ============================================================
# RETROSPECTIVE RANKING EVALUATION
# ============================================================

def evaluate_ranking(
    model_name,
    model,
    cleaned_history,
    evaluation_match_ids,
    top_k_values=(5, 6)
):
    """
    Retrospective ranking evaluation.

    For every historical match:
    1. Use ONLY history before that match.
    2. Generate candidate agents.
    3. Rank them.
    4. Check where the agent actually played appears.

    Matches where the played agent had never previously been
    used are skipped because that agent was not part of the
    known historical agent pool.
    """

    reciprocal_ranks = []

    hit_counts = {
        k: 0
        for k in top_k_values
    }

    evaluated = 0


    for match_id in evaluation_match_ids:

        matching_rows = cleaned_history[
            cleaned_history["Match id"]
            == match_id
        ]

        if matching_rows.empty:
            continue


        row_index = (
            matching_rows.index[0]
        )

        current_row = (
            cleaned_history.loc[
                row_index
            ]
        )


        history = (
            cleaned_history.iloc[
                :row_index
            ]
            .copy()
        )


        actual_agent = (
            current_row["Agent"]
        )

        current_map = (
            current_row["Map"]
        )


        # Can't evaluate first-time agent use because
        # recommender only considers previously used agents.
        if actual_agent not in history["Agent"].values:
            continue


        candidates = build_candidate_table(
            history=history,
            current_map=current_map,
            min_previous_matches=1
        )


        if candidates.empty:
            continue


        probabilities = (
            predict_win_probability(
                model_name,
                model,
                candidates[model_features]
            )
        )


        candidates[
            "probability"
        ] = probabilities


        candidates = (
            candidates
            .sort_values(
                "probability",
                ascending=False
            )
            .reset_index(drop=True)
        )


        matches = candidates.index[
            candidates["Agent"]
            == actual_agent
        ].tolist()


        if not matches:
            continue


        actual_rank = (
            matches[0] + 1
        )


        reciprocal_ranks.append(
            1 / actual_rank
        )


        for k in top_k_values:

            if actual_rank <= k:
                hit_counts[k] += 1


        evaluated += 1


    results = {
        "Model": model_name,
        "Evaluated Matches": evaluated,
        "MRR": (
            np.mean(reciprocal_ranks)
            if reciprocal_ranks
            else np.nan
        )
    }


    for k in top_k_values:

        results[f"Hit@{k}"] = (
            hit_counts[k] / evaluated
            if evaluated > 0
            else np.nan
        )


    return results

In [24]:
validation_match_ids = (
    validation_df["Match id"]
    .tolist()
)


validation_results = []


validation_results.append(
    evaluate_ranking(
        "Logistic Regression",
        logistic_model,
        cleaned_df,
        validation_match_ids
    )
)


validation_results.append(
    evaluate_ranking(
        "Random Forest",
        random_forest_model,
        cleaned_df,
        validation_match_ids
    )
)


validation_results.append(
    evaluate_ranking(
        "CatBoost",
        catboost_model,
        cleaned_df,
        validation_match_ids
    )
)


validation_ranking = pd.DataFrame(
    validation_results
)


validation_ranking

,Model,Evaluated Matches,MRR,Hit@5,Hit@6
0,Logistic Regression,43,0.286499,0.534884,0.627907
1,Random Forest,43,0.297370,0.581395,0.604651
2,CatBoost,43,0.356266,0.581395,0.697674


In [25]:
model_lookup = {
    "Logistic Regression":
        logistic_model,

    "Random Forest":
        random_forest_model,

    "CatBoost":
        catboost_model
}


validation_ranking = (
    validation_ranking
    .sort_values(
        by=[
            "Hit@6",
            "MRR"
        ],
        ascending=False
    )
    .reset_index(drop=True)
)


validation_ranking

,Model,Evaluated Matches,MRR,Hit@5,Hit@6
0,CatBoost,43,0.356266,0.581395,0.697674
1,Logistic Regression,43,0.286499,0.534884,0.627907
2,Random Forest,43,0.297370,0.581395,0.604651


In [26]:
selected_model_name = (
    validation_ranking.loc[
        0,
        "Model"
    ]
)


selected_model = (
    model_lookup[
        selected_model_name
    ]
)


print(
    "Selected model:",
    selected_model_name
)

Selected model: CatBoost


In [27]:
test_match_ids = (
    test_df["Match id"]
    .tolist()
)


test_ranking_result = (
    evaluate_ranking(
        selected_model_name,
        selected_model,
        cleaned_df,
        test_match_ids
    )
)


pd.DataFrame(
    [test_ranking_result]
)

,Model,Evaluated Matches,MRR,Hit@5,Hit@6
0,CatBoost,44,0.220851,0.727273,0.886364


## Ranking Evaluation Limitation

The historical dataset records the outcome only for the agent that was actually
selected in each match.

It does not contain counterfactual outcomes showing what would have happened if
a different agent had been selected in the same match.

Therefore, Hit@5, Hit@6, and MRR are used here as retrospective recommendation
proxies. They measure how highly the model ranks the agent that the player
actually selected using only information available before that match.

These metrics do not prove that the historically selected agent was objectively
the best possible agent.

In [28]:
test_results = []


for name, model in model_lookup.items():

    result = evaluate_ranking(
        name,
        model,
        cleaned_df,
        test_match_ids
    )

    test_results.append(result)


test_ranking = pd.DataFrame(
    test_results
)


test_ranking

c:\Users\HP\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
c:\Users\HP\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
c:\Users\HP\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
c:\Users\HP\AppData

,Model,Evaluated Matches,MRR,Hit@5,Hit@6
0,Logistic Regression,44,0.238672,0.590909,0.795455
1,Random Forest,44,0.233207,0.545455,0.818182
2,CatBoost,44,0.220851,0.727273,0.886364


In [29]:
CURRENT_MAP = "Haven"

In [30]:
recommendations = rank_agents(
    model_name=selected_model_name,
    model=selected_model,
    history=cleaned_df,
    current_map=CURRENT_MAP,
    top_k=6
)


recommendations

,Agent,Predicted Win Probability,agent_familiarity,agent_previous_matches,agent_previous_win_rate,agent_map_previous_matches,agent_map_previous_win_rate
Rank,,,,,,,
1,Killjoy,0.537213,Medium,30,0.633333,5,1.000000
2,Breach,0.535934,Medium,17,0.529412,2,0.500000
3,Skye,0.490709,Low,9,0.375000,1,1.000000
4,Sage,0.483262,High,161,0.464516,22,0.380952
5,Clove,0.469483,High,58,0.482143,11,0.600000
6,Deadlock,0.468698,Low,10,0.100000,0,NaN


In [31]:
display_recommendations = (
    recommendations.copy()
)


display_recommendations[
    "Predicted Win Probability"
] = (
    display_recommendations[
        "Predicted Win Probability"
    ]
    * 100
).round(2)


display_recommendations[
    "agent_previous_win_rate"
] = (
    display_recommendations[
        "agent_previous_win_rate"
    ]
    * 100
).round(2)


display_recommendations[
    "agent_map_previous_win_rate"
] = (
    display_recommendations[
        "agent_map_previous_win_rate"
    ]
    * 100
).round(2)


display_recommendations

,Agent,Predicted Win Probability,agent_familiarity,agent_previous_matches,agent_previous_win_rate,agent_map_previous_matches,agent_map_previous_win_rate
Rank,,,,,,,
1,Killjoy,53.72,Medium,30,63.33,5,100.0
2,Breach,53.59,Medium,17,52.94,2,50.0
3,Skye,49.07,Low,9,37.50,1,100.0
4,Sage,48.33,High,161,46.45,22,38.1
5,Clove,46.95,High,58,48.21,11,60.0
6,Deadlock,46.87,Low,10,10.00,0,NaN


In [32]:
def recommend_for_map(
    current_map,
    top_k=6
):

    recommendations = rank_agents(
        model_name=selected_model_name,
        model=selected_model,
        history=cleaned_df,
        current_map=current_map,
        top_k=top_k
    )

    result = (
        recommendations.copy()
    )

    result[
        "Predicted Win Probability"
    ] = (
        result[
            "Predicted Win Probability"
        ]
        * 100
    ).round(2)

    return result

In [33]:
recommend_for_map(
    "Haven"
)

,Agent,Predicted Win Probability,agent_familiarity,agent_previous_matches,agent_previous_win_rate,agent_map_previous_matches,agent_map_previous_win_rate
Rank,,,,,,,
1,Killjoy,53.72,Medium,30,0.633333,5,1.000000
2,Breach,53.59,Medium,17,0.529412,2,0.500000
3,Skye,49.07,Low,9,0.375000,1,1.000000
4,Sage,48.33,High,161,0.464516,22,0.380952
5,Clove,46.95,High,58,0.482143,11,0.600000
6,Deadlock,46.87,Low,10,0.100000,0,NaN


In [34]:
recommend_for_map(
    "Ascent"
)

,Agent,Predicted Win Probability,agent_familiarity,agent_previous_matches,agent_previous_win_rate,agent_map_previous_matches,agent_map_previous_win_rate
Rank,,,,,,,
1,Breach,53.62,Medium,17,0.529412,0,NaN
2,Killjoy,53.59,Medium,30,0.633333,7,0.428571
3,Deadlock,51.06,Low,10,0.100000,1,1.000000
4,Skye,48.91,Low,9,0.375000,3,0.000000
5,Sage,48.30,High,161,0.464516,22,0.523810
6,Clove,46.95,High,58,0.482143,5,0.600000


In [35]:
recommend_for_map(
    "Lotus"
)

,Agent,Predicted Win Probability,agent_familiarity,agent_previous_matches,agent_previous_win_rate,agent_map_previous_matches,agent_map_previous_win_rate
Rank,,,,,,,
1,Killjoy,53.72,Medium,30,0.633333,1,1.000000
2,Breach,53.59,Medium,17,0.529412,7,0.428571
3,Skye,48.94,Low,9,0.375000,1,0.000000
4,Sage,48.30,High,161,0.464516,25,0.320000
5,Clove,47.08,High,58,0.482143,3,0.666667
6,Deadlock,46.87,Low,10,0.100000,0,NaN


In [ ]:
recommend_for_map(
    "Corrode"
)

,Agent,Predicted Win Probability,agent_familiarity,agent_previous_matches,agent_previous_win_rate,agent_map_previous_matches,agent_map_previous_win_rate
Rank,,,,,,,
1,Breach,53.62,Medium,17,0.529412,0,NaN
2,Killjoy,53.59,Medium,30,0.633333,1,0.0
3,Skye,48.94,Low,9,0.375000,0,NaN
4,Sage,48.43,High,161,0.464516,1,1.0
5,Clove,47.10,High,58,0.482143,1,1.0
6,Deadlock,46.87,Low,10,0.100000,0,NaN


: 